In [ ]:
from pathlib import Path

RESULTS_PATH = Path("./results")
CONFIDENCE_PATH = Path("./confidence_results") # path to Data.zip ...

ASER_RESULTS_PATH = RESULTS_PATH / "assessor-error_risk"
ASAR_RESULTS_PATH = RESULTS_PATH / "assessor-answerability_risk"
BB_RESULTS_PATH = RESULTS_PATH / "black-box"
GB_RESULTS_PATH = RESULTS_PATH / "grey-box"

for path in (
    ASER_RESULTS_PATH,
    ASAR_RESULTS_PATH,
    BB_RESULTS_PATH,
    GB_RESULTS_PATH,
):
    path.mkdir(parents=True, exist_ok=True)
    (path / "IID").mkdir(parents=True, exist_ok=True)
    (path / "OOD").mkdir(parents=True, exist_ok=True)

# Results

In [ ]:
from itertools import product
import pandas as pd
import numpy as np
from modeling.utils import LabelEncoderWithReject
from modeling.evaluation import RejectionAnalyzer


def yield_combinations(**spaces):
    keys = list(spaces.keys())
    values = list(spaces.values())
    for combination in product(*values):
        yield dict(zip(keys, combination))


def read_confidence_csv(distribution, engine, benchmark, feature=None, variant=None, assessor=True):
    path = CONFIDENCE_PATH / distribution / engine / benchmark 
    fname = ""
    fname += "asr_" if assessor else ""
    fname += f"{feature}_" if feature else ""
    fname += f"{variant}_" if variant else ""
    fname = fname.rstrip('_') + ".csv"
    path = path / fname
    if not path.exists():
        return None
    df = pd.read_csv(path).dropna(subset=['confidence', 'outcome'])
    return df


def build_probas(confidence, le):
    risk_score = 1 - (confidence / 100.0)
    probs = np.zeros((len(risk_score), 3))
    probs[:, le.transform(['INCORRECT'])[0]] = risk_score
    return probs


def evaluate_method(y_true, probs, le):
    ra = RejectionAnalyzer(label_encoder=le)
    analyzer = ra(y_true=y_true, probs=probs)
    return analyzer.evaluate()


def metrics_to_df(metrics_dict):
    data = {'partition': [], 'metric': [], 'value': [], 'margin': []}
    for metric, value in metrics_dict.items():
        if metric.endswith('_margin'):
            continue
        data['partition'].append('val')
        data['metric'].append(metric)
        data['value'].append(value)
        data['margin'].append(metrics_dict.get(metric + '_margin', None))
    return pd.DataFrame(data)


def add_constants_to_df(df, **constants):
    for key, value in constants.items():
        df[key] = value
    return df

In [ ]:
from tqdm import tqdm

BENCHMARKS = [
    'addition',
    'anagram',
    'locality',
    'science',
    'algebra',
    'arithmetic',
    'combinatory',
    'geometry',
    'logic_puzzles',
    'number_theory'
]

ENGINES = [
    'gemini-3.1-pro-preview',
    'GPT5.2',
    'llama3v3chat-70B',
    'Qwen2.5_72B'
]

FEATURES = [
    'difficulty',
    'embedding',
    'demands',
    'difficulty+embedding',
    'difficulty+demands',
    'embedding+demands',
    'difficulty+embedding+demands'
]

ASER2ASAR_FEATURE_MAPPING ={
    'difficulty': 'diff',
    'embedding': 'emb',
    'demands': 'adele',
    'difficulty+embedding': 'emb+diff',
    'difficulty+demands': 'diff+adele',
    'embedding+demands': 'emb+adele',
    'difficulty+embedding+demands': 'emb+diff+adele'
}

ASAR2ASER_FEATURE_MAPPING = {
    v: k for k, v in ASER2ASAR_FEATURE_MAPPING.items()
}

BB_VARIANTS = [
    'vanilla',
    'CoT',
    'self_probing',
    'multi_step',
    'top_k',
    'self_consistency'
]

GB_VARIANTS = [
    'SAR',
    'semantic_energy'
]

METHOD_SPACES = {
    'ASAR': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'variant': BB_VARIANTS+GB_VARIANTS,
        'feature': [ASER2ASAR_FEATURE_MAPPING[f] for f in FEATURES]
    },
    'ASER': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'feature': FEATURES},
    'BB': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'variant': BB_VARIANTS
    },
    'GB': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'variant': GB_VARIANTS
    }
}

def build_results(method, distribution):
    le = LabelEncoderWithReject()
    le.fit(['CORRECT', 'INCORRECT', 'AVOIDANCE'])
    dfs = []
    for combination in tqdm(yield_combinations(
        **METHOD_SPACES[method]),
        desc=f"{method} - {distribution}"
    ):
        df = read_confidence_csv(
            distribution,
            **combination,
            assessor=(method in ['ASAR', 'ASER'])
        )
        if df is None:
            continue
        probs = build_probas(df['confidence'].values, le)
        y_true = le.transform(df['outcome'])
        metrics = evaluate_method(y_true, probs, le)
        mdf = metrics_to_df(metrics)
        mdf = add_constants_to_df(
            mdf,
            engine=combination['engine'],
            benchmark=combination['benchmark'],
            variant=combination.get('variant', None),
            features=ASAR2ASER_FEATURE_MAPPING.get(
                combination.get('feature', None),
                combination.get('feature', None)
            ), 
            num_instances=len(df)
        )
        dfs.append(mdf)
    results = pd.concat(dfs, ignore_index=True)
    results.loc[(results['metric'] == 'FOC-AUC') & (results['value'] == 0), 'value'] = np.nan
    tgt_cols = ['engine', 'benchmark', 'variant', 'features', 'metric', 'value', 'margin', 'num_instances']
    return results[tgt_cols]

def save_results(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

In [ ]:
METHODS = [
    ('ASER', ASER_RESULTS_PATH),
    ('ASAR', ASAR_RESULTS_PATH),
    ('BB', BB_RESULTS_PATH),
    ('GB', GB_RESULTS_PATH)
]
DISTRIBUTIONS = ['IID', 'OOD']

for method, results_path in METHODS:
    for distribution in DISTRIBUTIONS:
        if distribution == 'OOD' and method in ['BB', 'GB']:
            continue
        res = build_results(method, distribution)
        if method in ['ASAR', 'ASER']:
            save_results(res, results_path / distribution / "metrics.csv")
        else:
            save_results(res, results_path / "metrics.csv")

# Metrics

In [ ]:
def get_metrics(distribution):
    dfs = []
    for method, method_path in METHODS:
        if method in ['ASER', 'ASAR']:
            method_path /=  distribution 
        df = pd.read_csv(method_path / "metrics.csv")
        df.insert(0, 'method', method)
        dfs.append(df)
    metrics_df = pd.concat(dfs, ignore_index=True)
    metrics_df['method'] = pd.Categorical(metrics_df['method'], categories=[m[0] for m in METHODS], ordered=True)
    metrics_df['engine'] = pd.Categorical(metrics_df['engine'], categories=ENGINES, ordered=True)
    metrics_df['benchmark'] = pd.Categorical(metrics_df['benchmark'], categories=BENCHMARKS, ordered=True)
    metrics_df['features'] = pd.Categorical(metrics_df['features'].fillna('-'), categories=FEATURES+['-'], ordered=True)
    metrics_df['variant'] = pd.Categorical(metrics_df['variant'].fillna('-'), categories=BB_VARIANTS+GB_VARIANTS+['-'], ordered=True)
    return metrics_df

metrics_iid = get_metrics('IID')
metrics_ood = get_metrics('OOD')
print(f"IID metrics: {metrics_iid.shape[0]} rows")
print(f"OOD metrics: {metrics_ood.shape[0]} rows")

In [ ]:
METHOD_FORMATTER = {
    'ASER': r"$\mathcal{A_{ER}}$",
    'ASAR': r"$\mathcal{A_{AR}}$",
    'BB': r"Black-Box",
    'GB': r"Grey-Box"
}

VARIANT_FORMATTER = {
    'vanilla': r"\textit{Vanilla}",
    'CoT': r"\textit{CoT}",
    'self_probing': r"\textit{Self-Prob}",
    'multi_step': r"\textit{Multi-Step}",
    'top_k': r"\textit{Top-K}",
    'self_consistency': r"\textit{Self-Cons}",
    'SAR': r"\textit{SAR}",
    'semantic_energy': r"\textit{Sema-Energy}"
}

METRIC_FORMATTER = {
        'original-error': r'\textbf{Orig. Error}$\downarrow$',
        "error@MAX-ERG": r"\textbf{Err}$\downarrow$",
        "PVR@MAX-ERG": r"\textbf{Cov}$\uparrow$",
        "SAR@MAX-ERG": r"\textbf{SAR}$\uparrow$",
        "ERR@MAX-ERG": r"\textbf{URR}$\uparrow$",
}

def format_feats(feat_str):
    if pd.isna(feat_str):
        return '-'
    feat_str = feat_str.replace("difficulty", r"\textit{Diff}")
    feat_str = feat_str.replace("embedding", r"\textit{Emb}")
    feat_str = feat_str.replace("demands", r"\textit{DeLeAn}")
    return feat_str

def format_engine(engine_str):
    return f"\modelt{engine_str}"

def bold_max_per_method_and_benchmark(df, subset=None, direction={}, method_col='method'):
    df_out = df.astype(object).copy()
    for method in df.index.get_level_values(method_col).unique():
        method_slice = df.xs(method, level=method_col)
        for col in df.columns:
            if subset is not None and col not in subset:
                continue
            if direction.get(col, 'max') == 'max':
                val = method_slice[col].max()
            else:
                val = method_slice[col].min()
            mask = (
                (df.index.get_level_values(method_col) == method) &
                (df[col] == val)
            )
            df_out.loc[mask, col] = (
                df_out.loc[mask, col]
                .astype(float)
                .map(lambda x: f"\\textbf{{{x:.3f}}}")
            )
    for col in df_out.columns:
        df_out[col] = df_out[col].apply(
            lambda x: f"{x:.3f}" if isinstance(x, float) else x
        )
    return df_out

#### Table 1: Summary table @ ORP

In [ ]:
import pandas as pd

def format_summary_table(summary_df, distribution='IID'):
    formatted = summary_df.copy()
    def format_method_id(method_id):
        method, variant, features = method_id.split('|')
        method = METHOD_FORMATTER.get(method.strip(), method.strip())
        variant = VARIANT_FORMATTER.get(variant.strip(), variant.strip())
        features = format_feats(features.strip())
        return f"{method} | {variant} | {features}"
    formatted['method_id'] = formatted['method_id'].apply(format_method_id)
    formatted['engine'] = formatted['engine'].apply(format_engine)
    formatted.columns = [
        METRIC_FORMATTER.get(col, col) if col not in ['model', 'method'] else col
        for col in formatted.columns
        
    ]
    caption = f"Summary table of best methods per model based on median %s ERG@MAX-ERG across benchmarks."
    if distribution == 'IID':
        caption = caption % "in distribution"
    else:
        caption = caption % "out of distribution"
    formatted = formatted.to_latex(
        float_format="%.3f",
        na_rep="nan",
        caption=caption,
        label="tab:summary_table_best_methods_per_model",
        bold_rows=True,
        escape=False,
        index=False
    )
    return formatted

def summary_table(df, distribution='IID'):
    target_metrics = [
        "original-error",
        "ACC@MAX-ERG",
        "PVR@MAX-ERG",
        "SAR@MAX-ERG",
        "ERR@MAX-ERG",
        "ERG@MAX-ERG",
        "FOC-AUC"
    ]

    df_filt = df[df["metric"].isin(target_metrics)].copy()

    df_filt["method_id"] = (
        df_filt["method"].astype(str) + " | " +
        df_filt["variant"].astype(str) + " | " +
        df_filt["features"].astype(str)
    )

    pivot = (
        df_filt
        .pivot_table(
            index=["engine", "benchmark", "method_id"],
            columns="metric",
            values="value",
            aggfunc="median"
        )
        .reset_index()
    )

    mean_by_method = (
        pivot
        .groupby(["engine", "method_id"])
        .median(numeric_only=True)
        .reset_index()
    )

    best_methods = (
        mean_by_method
        .sort_values("ERG@MAX-ERG", ascending=False)
        .groupby("engine", as_index=False)
        .first()
        .sort_values("engine")
    )

    best_methods['error@MAX-ERG'] = 1 - best_methods['ACC@MAX-ERG']
    summary = best_methods[
        [
            "engine", "method_id", 
            "original-error", "error@MAX-ERG", 
            "PVR@MAX-ERG", "SAR@MAX-ERG", "ERR@MAX-ERG"
        ]
    ]
    return summary, format_summary_table(summary, distribution)

In [ ]:
summary_iid, summary_iid_formatted = summary_table(metrics_iid, distribution='IID')
summary_iid.round(3)

In [ ]:
print(summary_iid_formatted)

In [ ]:
summary_ood, summary_ood_formatted = summary_table(metrics_ood, distribution='OOD')
summary_ood.round(3)

In [ ]:
print(summary_ood_formatted)

#### Table 2 - Median FOC-AUC results across models per benchmark for selected methods and feature sets.

In [ ]:
# mean across engines and benchmarks per method, variant and feature set

TABLE2_METHOD_FORMATTER = {
    'ASER': r"\assAlv",
    'ASAR': r"\assKex",
    'BB': r"Black-Box",
    'GB': r"Grey-Box",
}
TABLE2_VARIANT_FORMATTER = {**VARIANT_FORMATTER, 'semantic_energy': r"\textit{Sem-Energy}"}


def render_foc_table(df, caption, label, comment=None, clearpage=False,
                     column_names=None, bold_subset=None, bold_direction=None):
    """Render a (method, variant, features) x columns table as LaTeX.

    Uses table* + \\resizebox (sn-jnl wraps plain `table` in threeparttable,
    which breaks \\resizebox around a tabular).
    column_names: optional {column: LaTeX header}; other columns get '_' escaped.
    bold_subset / bold_direction: passed to bold_max_per_method_and_benchmark.
    """
    # bold best value per method (level must still be called 'method' here)
    bolded = bold_max_per_method_and_benchmark(
        df.round(3), subset=bold_subset, direction=bold_direction or {}
    )

    column_names = column_names or {}
    columns = [column_names.get(column, str(column).replace('_', r'\_')) for column in bolded.columns]
    n_cols = 3 + len(columns)
    header = [r"$\mathbf{\mathfrak{m}}$", "Prompt", r"$\phi(\mu)$"] + columns

    # group rows (keeping their order) by method, then by variant
    groups = []  # [(method, [(variant, [(features, values), ...]), ...]), ...]
    for (method, variant, features), row in bolded.iterrows():
        if not groups or groups[-1][0] != method:
            groups.append((method, []))
        variants = groups[-1][1]
        if not variants or variants[-1][0] != variant:
            variants.append((variant, []))
        variants[-1][1].append((features, row.tolist()))

    def multirow(n, text):
        return rf"\multirow[t]{{{n}}}{{*}}{{{text}}}" if n > 1 else text

    body = []
    for m_idx, (method, variants) in enumerate(groups):
        method_txt = TABLE2_METHOD_FORMATTER.get(method, method)
        n_method = sum(len(rows) for _, rows in variants)
        for v_idx, (variant, rows) in enumerate(variants):
            variant_txt = TABLE2_VARIANT_FORMATTER.get(variant, variant)
            for r_idx, (features, values) in enumerate(rows):
                c1 = multirow(n_method, method_txt) if (v_idx == 0 and r_idx == 0) else ""
                c2 = multirow(len(rows), variant_txt) if r_idx == 0 else ""
                cells = [c1, c2, format_feats(features)] + [str(v) for v in values]
                body.append(" & ".join(cells) + r" \\")
            if v_idx < len(variants) - 1:
                body.append(rf"\cmidrule(l){{2-{n_cols}}}")
        if m_idx < len(groups) - 1:
            body.append(r"\midrule")

    lines = [comment, ""] if comment else []
    lines += [
        r"\begin{table*}[!ht]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\setlength{\tabcolsep}{6pt}",
        r"\renewcommand{\arraystretch}{0.9}",
        r"\resizebox{\linewidth}{!}{%",
        r"\begin{tabular}{ccr" + "c" * len(columns) + "}",
        r"\toprule",
        " & ".join(header) + r" \\",
        r"\midrule",
        *body,
        r"\bottomrule",
        r"\end{tabular}",
        r"}",
        r"\end{table*}",
    ]
    if clearpage:
        # flush the float here so following [H] figures don't overtake it
        lines.append(r"\clearpage")
    return "\n".join(lines) + "\n"


def format_table2(df, distribution='IID', is_selection=False):
    dist_txt = "In distribution" if distribution == 'IID' else "Out of distribution"
    caption = f"{dist_txt} mean FOC-AUC results across models per benchmark."
    label = "tab:foc_auc_mean_across_benchmarks{}_{}".format(
        "_selected" if is_selection else "", distribution.lower()
    )
    # the full (appendix) table gets \clearpage; the selected main-text one does not
    return render_foc_table(df, caption, label, comment=r"%% [MEAN]", clearpage=not is_selection)


def table2_long(metrics, distribution='ID'):
    long = (
        metrics[(metrics['metric'] == 'FOC-AUC')]
            .groupby(['method', 'variant', 'features', 'benchmark'], observed=True)['value'].mean()
            .reset_index()
            .pivot(index=['method', 'variant', 'features'], columns='benchmark', values='value')
    )

    long['mean'] = long.mean(axis=1)
    return long, format_table2(long, distribution=distribution)

def table2_selection(long, distribution='IID'):
    selection = [
        ('ASER', '-', 'embedding+demands'),
        ('ASER', '-', 'difficulty+embedding+demands'),
        ('ASAR', 'CoT', 'difficulty+embedding+demands'),
        ('ASAR', 'self_probing', 'difficulty+embedding+demands'),
        ('BB', 'CoT', '-'),
        ('BB', 'self_consistency', '-'),
        ('GB', 'SAR', '-'),
    ]
    selection = long.loc[selection, :]
    return selection, format_table2(selection, distribution=distribution, is_selection=True)

In [ ]:
table2_long_iid, table2_long_iid_formatted = table2_long(metrics_iid, distribution='IID')
table2_selection_iid, table2_selection_iid_formatted = table2_selection(table2_long_iid, distribution='IID')
table2_selection_iid.round(3)

In [ ]:
print(table2_selection_iid_formatted)

In [ ]:
table2_long_iid.round(3)

In [ ]:
print(table2_long_iid_formatted)

In [ ]:
table2_long_ood, table2_long_ood_formatted = table2_long(metrics_ood, distribution='OOD')
table2_selection_ood, table2_selection_ood_formatted = table2_selection(table2_long_ood, distribution='OOD')
table2_selection_ood.round(3)

In [ ]:
print(table2_selection_iid_formatted)

In [ ]:
table2_long_ood.round(3)

In [ ]:
print(table2_long_ood_formatted)

#### Supplemenetary Tables A - FOC-AUC results per engine across tasks, prompts and asessor feature sets.

In [ ]:
import re


def suptableA(metrics, engine):
    table = (
        metrics[metrics['metric'] == 'FOC-AUC']
            .pivot(index=['engine', 'method', 'variant', 'features'], columns='benchmark', values='value')
            .xs(engine, level='engine')
    )
    return table


def format_suptableA(table, engine, distribution='IID'):
    dist_txt = "In distribution" if distribution == 'IID' else "Out of distribution"
    engine_tex = engine.replace('_', r'\_')
    caption = (
        rf"{dist_txt} FOC-AUC results for \modelt{{{engine_tex}}} across tasks, prompts and assessor feature sets. "
        "Rows correspond to assessor configurations and prompting strategies, columns to benchmark tasks."
    )
    engine_slug = re.sub(r'[^A-Za-z0-9]+', '_', engine).strip('_').lower()
    label = f"tab:foc_auc_{distribution.lower()}_{engine_slug}"
    return render_foc_table(table, caption, label, clearpage=True)

In [ ]:
distribution = 'IID'
for engine in ENGINES:
    print(f"%% [{engine.upper()}]\n")
    table = suptableA(metrics_iid, engine)
    formatted = format_suptableA(table, engine, distribution=distribution)
    print(formatted)
    

In [ ]:
distribution = 'OOD'
for engine in ENGINES:
    print(f"%% [{engine.upper()}]\n")
    table = suptableA(metrics_ood, engine)
    formatted = format_suptableA(table, engine, distribution=distribution)
    print(formatted)
    

#### Supplementary Tables B - summary of rejection metrics at ORP for `engine` on `benchmark`

In [ ]:
import re


def summary_long(df):
    target_metrics = [
        "original-error",
        "ACC@MAX-ERG",
        "PVR@MAX-ERG",
        "SAR@MAX-ERG",
        "ERR@MAX-ERG",
    ]
    df_filt = df[df["metric"].isin(target_metrics)].copy()
    table = (
        df_filt
        .pivot_table(
            index=["engine", "benchmark", "method", "variant", "features"],
            columns="metric",
            values="value",
            aggfunc="mean"
        )
        .reset_index()
    )
    table = table[
        [
            "engine",
            "benchmark",
            "method",
            "variant",
            "features",
            "original-error",
            "ACC@MAX-ERG",
            "PVR@MAX-ERG",
            "SAR@MAX-ERG",
            "ERR@MAX-ERG",
        ]
    ]
    table['error@MAX-ERG'] = 1 - table['ACC@MAX-ERG']
    return table


# column order + headers, matching the ORP table in the main text
SUPB_COLUMNS = {
    'original-error': r"\textbf{Orig. Err}$\downarrow$",
    'error@MAX-ERG': r"\textbf{Err}$\downarrow$",
    'PVR@MAX-ERG': r"\textbf{Cov}$\uparrow$",
    'SAR@MAX-ERG': r"\textbf{S-Acc}$\uparrow$",
    'ERR@MAX-ERG': r"\textbf{U-Rej}$\uparrow$",
}
# Orig. Err is the base model's error (same for every method), so it is not bolded
SUPB_BOLD_DIRECTION = {
    'error@MAX-ERG': 'min',
    'PVR@MAX-ERG': 'max',
    'SAR@MAX-ERG': 'max',
    'ERR@MAX-ERG': 'max',
}


def format_suptableB_long(summary_long, distribution='IID'):
    """One table per (engine, benchmark) subset of summary_long()."""
    if summary_long.empty:
        return "% (no data for this engine/benchmark)\n"
    engines = summary_long['engine'].unique()
    benchmarks = summary_long['benchmark'].unique()
    assert len(engines) == 1 and len(benchmarks) == 1, "pass one engine/benchmark subset"
    engine, benchmark = str(engines[0]), str(benchmarks[0])

    table = summary_long.set_index(['method', 'variant', 'features'])[list(SUPB_COLUMNS)]

    in_dist = distribution.upper() in ('IID', 'ID')
    dist_txt = "In distribution" if in_dist else "Out of distribution"
    engine_tex = engine.replace('_', r'\_')
    benchmark_tex = benchmark.replace('_', r'\_')
    caption = (
        rf"{dist_txt} rejection metrics at the optimal threshold for "
        rf"\modelt{{{engine_tex}}} on \modelt{{{benchmark_tex}}}."
    )
    slug = lambda s: re.sub(r'[^A-Za-z0-9]+', '_', s).strip('_').lower()
    label = f"tab:orp_{'iid' if in_dist else 'ood'}_{slug(engine)}_{slug(benchmark)}"

    return render_foc_table(
        table, caption, label,
        column_names=SUPB_COLUMNS,
        bold_subset=list(SUPB_BOLD_DIRECTION),
        bold_direction=SUPB_BOLD_DIRECTION,
        clearpage=True,
    )

In [ ]:
summary_iid = summary_long(metrics_iid)
for engine in ENGINES:
    for benchmark in BENCHMARKS:
        print(f"%% [{engine.upper()} - {benchmark}]\n")
        subset = summary_iid[
            (summary_iid['engine'] == engine) &
            (summary_iid['benchmark'] == benchmark)
        ]
        print(format_suptableB_long(subset))

In [ ]:
summary_ood = summary_long(metrics_ood)
for engine in ENGINES:
    for benchmark in BENCHMARKS:
        print(f"%% [{engine.upper()} - {benchmark}]\n")
        subset = summary_ood[
            (summary_ood['engine'] == engine) &
            (summary_ood['benchmark'] == benchmark)
        ]
        print(format_suptableB_long(subset, distribution='OOD'))